In [0]:
# Silver Layer - Limpeza e Padronização
from pyspark.sql.functions import col, to_timestamp, trim, upper, current_timestamp
from pyspark.sql.types import DoubleType, IntegerType

# Leitura da Bronze
df_bronze = spark.read.table("techdados.bronze.vendas")

# Transformações Silver
df_silver = (
    df_bronze
    # Tipagem correta
    .withColumn("data_venda", to_timestamp(col("data_venda"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("quantidade", col("quantidade").cast(IntegerType()))
    .withColumn("preco_unitario", col("preco_unitario").cast(DoubleType()))
    
    # Limpeza e padronização
    .withColumn("produto", trim(col("produto")))
    .withColumn("loja", upper(trim(col("loja"))))
    .withColumn("status", trim(col("status")))
    
    # Filtra registros inválidos
    .filter(col("status") != "cancelado")
    .filter(col("quantidade") > 0)
    
    # Remove duplicatas
    .dropDuplicates(["id"])
    
    # Adiciona campo calculado
    .withColumn("valor_total", col("quantidade") * col("preco_unitario"))
        
    # Remove colunas de metadados da bronze
    .drop("_ingestion_timestamp", "_source_file")
)

# Salva como Delta na camada Silver (SCD Type 1 - Overwrite)
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("techdados.silver.vendas")
)